In [3]:
import sys
import os
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType

# Đảm bảo Python nhận diện thư mục hiện tại (src) để import spark_session
sys.path.append(os.path.abspath("."))


In [4]:
import spark_session

# Tự động lấy đối tượng spark hoặc hàm khởi tạo bên trong spark_session.py
if hasattr(spark_session, "get_spark"):
    spark = spark_session.get_spark("HM_Clean_Huy")
elif hasattr(spark_session, "create_spark_session"):
    spark = spark_session.create_spark_session("HM_Clean_Huy")
elif hasattr(spark_session, "spark"):
    spark = spark_session.spark
else:
    raise AttributeError("Kiểm tra tên hàm tạo Spark bên trong file src/spark_session.py!")

print("Đã khởi tạo Spark version:", spark.version)

Đã khởi tạo Spark version: 4.2.0


In [5]:
# đọc dữ liệu từ file CSV
df_art = spark.read.parquet("../data/raw/article.parquet")
df_cust = spark.read.parquet("../data/raw/customer.parquet")
df_trans = spark.read.parquet("../data/processed/transactions.parquet")

stats = []
def log_step(step_name, df):
    row_cnt = df.count()
    cust_cnt = df.select("customer_id").distinct().count()
    stats.append({
        "Buoc": step_name,
        "So_dong": row_cnt,
        "So_khach_hang": cust_cnt
    })
    print(f"[{step_name}] Số dòng: {row_cnt:,} | Số khách hàng: {cust_cnt:,}")

log_step("0. Dữ liệu gốc ban đầu", df_trans)

[0. Dữ liệu gốc ban đầu] Số dòng: 15,453,651 | Số khách hàng: 1,007,704


In [6]:
# chuẩn hóa dữ liệu
df_trans = (
    df_trans
    .withColumn("t_dat", F.to_date(F.col("t_dat")))
    .withColumn("price", F.col("price").cast(DoubleType()))
    .withColumn("sales_channel_id", F.col("sales_channel_id").cast(LongType()))
)

In [7]:
# Loại dòng thiếu khóa chính và ngày giao dịch
df_trans = df_trans.dropna(subset=["customer_id", "article_id", "t_dat"])
log_step("1. Sau khi loại bỏ null khóa chính & ngày", df_trans)

# Kiểm tra và loại khóa mồ côi bằng left_anti / inner join
orphan_cust = df_trans.join(df_cust, on="customer_id", how="left_anti").count()
orphan_art = df_trans.join(df_art, on="article_id", how="left_anti").count()
print(f"-> Số dòng khóa mồ côi: customer_id = {orphan_cust:,}, article_id = {orphan_art:,}")

df_trans = (
    df_trans
    .join(df_cust.select("customer_id"), on="customer_id", how="inner")
    .join(df_art.select("article_id"), on="article_id", how="inner")
)
log_step("2. Sau khi loại bỏ khóa mồ côi", df_trans)

[1. Sau khi loại bỏ null khóa chính & ngày] Số dòng: 15,453,651 | Số khách hàng: 1,007,704
-> Số dòng khóa mồ côi: customer_id = 0, article_id = 0
[2. Sau khi loại bỏ khóa mồ côi] Số dòng: 15,453,651 | Số khách hàng: 1,007,704


In [8]:
# Loại bỏ giao dịch có price <= 0 hoặc NaN/Null
df_trans = df_trans.filter((F.col("price").isNotNull()) & (~F.isnan("price")) & (F.col("price") > 0))
log_step("3. Sau khi lọc price > 0", df_trans)

[3. Sau khi lọc price > 0] Số dòng: 15,453,651 | Số khách hàng: 1,007,704


In [9]:
# Thống kê dòng trùng lặp hoàn toàn 
total_rows = df_trans.count()
distinct_rows = df_trans.dropDuplicates().count()
dup_rows = total_rows - distinct_rows
print(f"-> Số dòng trùng lặp hoàn toàn (được giữ lại): {dup_rows:,} ({dup_rows/total_rows*100:.2f}%)")

-> Số dòng trùng lặp hoàn toàn (được giữ lại): 1,417,034 (9.17%)


In [10]:
# Làm sạch bảng customer 
median_age = df_cust.filter((F.col("age") >= 12) & (F.col("age") <= 100)).approxQuantile("age", [0.5], 0.01)[0]
print(f"-> Tuổi trung vị hợp lệ: {median_age}")

df_cust_clean = (
    df_cust
    .withColumn("is_age_missing", F.when((F.col("age").isNull()) | (F.col("age") < 12) | (F.col("age") > 100), 1).otherwise(0))
    .withColumn("age_clean", F.when(F.col("is_age_missing") == 1, median_age).otherwise(F.col("age")))
    .withColumn("club_member_status", F.coalesce(F.col("club_member_status"), F.lit("UNKNOWN")))
    .withColumn("fashion_news_frequency", F.coalesce(F.col("fashion_news_frequency"), F.lit("NONE")))
)

-> Tuổi trung vị hợp lệ: 32.0


In [11]:
# Lưu dữ liệu sạch như mục data/clean
os.makedirs("../data/clean", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

df_trans.write.mode("overwrite").parquet("../data/clean/transactions_clean.parquet")
df_cust_clean.write.mode("overwrite").parquet("../data/clean/customer_clean.parquet")
df_art.write.mode("overwrite").parquet("../data/clean/article_clean.parquet")

In [12]:
# xuất bảng thống kê làm sạch 
df_stats = pd.DataFrame(stats)
df_stats.to_csv("../reports/thong_ke_lam_sach.csv", index=False, encoding="utf-8-sig")
df_stats

,Buoc,So_dong,So_khach_hang
0,0. Dữ liệu gốc ban đầu,15453651,1007704
1,1. Sau khi loại bỏ null khóa chính & ngày,15453651,1007704
2,2. Sau khi loại bỏ khóa mồ côi,15453651,1007704
3,3. Sau khi lọc price > 0,15453651,1007704
